In [ ]:
"""LiDAR 点群からの地面高さ推定アルゴリズムの検証スクリプト.

TypeScript 実装 (groundEstimation.ts) と同一ロジックの Python 移植版.
実データに対して複数の位置で推定を実行し、結果を可視化して妥当性を確認する.

依存: requests, numpy, matplotlib
    pip install requests numpy matplotlib
"""
import requests
import numpy as np
import matplotlib.pyplot as plt

# ── 設定 ─────────────────────────────────────────────────────────────────────

API_HOST = 'http://192.168.11.7'
API_PORT = 8000
SAMPLE_TOKEN = '3e8750f331d7499e9b5123e9eb70f2e2'

# アルゴリズムパラメータ (TypeScript 版と同一)
SEARCH_RADII       = [3.0, 5.0, 7.0]  # 半径探索の段階 (m)
MIN_POINTS         = 20               # 有効とみなす最低点数
BIN_SIZE           = 0.1              # ヒストグラムビン幅 (m)
BIN_RATIO_THRESH   = 0.10             # 有効ビンの点数閾値 (収集点数に対する比率) → いったん不使用とする
REFINE_HALF_WIDTH  = 0.15             # 精密化時にビン中心から拾う半幅 (m)
LOWER_MARGIN       = 1.0              # -LiDAR高さより下に許容する範囲 (m)
UPPER_MARGIN       = 1.5              # -LiDAR高さより上に許容する範囲 (m)
MAX_DEVIATION      = 1.0              # prior からの最大許容乖離 (m)

# ── アルゴリズム本体 ──────────────────────────────────────────────────────────

def estimate_ground_z(
    points: np.ndarray,
    target_xy: tuple[float, float],
    sensor_height: float,
    ego_z: float,
    verbose: bool = False,
) -> tuple[float, str]:
    """指定位置周辺の地面高さ (センサー座標系 z) を推定する.

    Args:
        points:        (N, 3+) 点群 [x, y, z, ...]
        target_xy:     地面高さを求めたい水平位置 (x, y)
        sensor_height: LIDAR の取り付け高さ (calibrated_sensor.translation[2]) (m)
        ego_z:         自車の地面からの高さ (ego_pose.translation[2]) (m)
        verbose:       途中経過を表示

    Returns:
        (ground_z, method): 推定値と、どの経路で決まったかのラベル
            method: 'estimated(R=..)' | 'prior(insufficient)' | 'prior(deviation)'
    """
    # LIDAR_TOP の取り付け高さ (m) = calibrated_sensor.translation[2] の負の値が prior として使われる
    z_min = -sensor_height - LOWER_MARGIN
    z_max = -sensor_height + UPPER_MARGIN
    tx, ty = target_xy

    # 事前 z フィルタ (全半径で共通なので先に 1 回だけ)
    z_ok = (points[:, 2] >= z_min) & (points[:, 2] <= z_max)
    pts = points[z_ok]

    dx = pts[:, 0] - tx
    dy = pts[:, 1] - ty
    dist2 = dx * dx + dy * dy

    for radius in SEARCH_RADII:
        zs = pts[dist2 <= radius * radius, 2]

        # ヒストグラム化
        bin_count = int(np.ceil((z_max - z_min) / BIN_SIZE))
        bins, bin_edges = np.histogram(zs, bins=bin_count, range=(z_min, z_max))

        # 「十分な点数を持つ最も低いビン」を選択　→ 不使用
        # threshold = max(3, len(zs) * BIN_RATIO_THRESH)
        # ground_bin_idx = -1
        # for i in range(bin_count):
        #     if bins[i] >= threshold:
        #         ground_bin_idx = i
        #         break
        # if ground_bin_idx == -1:
        #     continue

        # 最も高さが高いビンを選択
        ground_bin_idx = np.argmax(bins)

        # ビン中心 ±REFINE_HALF_WIDTH の点の中央値で精密化
        bin_center = z_min + (ground_bin_idx + 0.5) * BIN_SIZE
        refined = zs[np.abs(zs - bin_center) <= REFINE_HALF_WIDTH]
        if verbose:
            print(f'  R={radius}m: {len(refined)} 点 (精密化後)')
        # 最低点数の妥当性チェック
        if len(refined) < MIN_POINTS:
            continue  # 半径を拡大して再試行
        
        ground_z = float(np.median(refined)) # ここで求められた高さはLiDAR座標系のz値であり、prior_groundと比較する際にはLiDARの取り付け高さを考慮する必要がある

        if verbose:
            print(f'  選択ビン: [{bin_edges[ground_bin_idx]:.2f}, '
                  f'{bin_edges[ground_bin_idx + 1]:.2f}] ({bins[ground_bin_idx]} 点, ')
            print(f'  精密化: {len(refined)} 点の median = {ground_z:.3f}')

        # LiDARの取り付け高さからの乖離妥当性チェック
        if abs(ground_z - (-sensor_height)) > MAX_DEVIATION:
            continue  # prior から乖離しすぎ → 次の半径で再試行

        # 見つかった高さにsensor_heightとego_zを加算して、グローバル座標系に変換
        ground_z_global = ground_z + sensor_height + ego_z

        return ground_z_global, f'estimated(R={radius})'

    return ego_z, 'prior(insufficient)'


# ── データ取得 ────────────────────────────────────────────────────────────────

def fetch_pointcloud(url: str) -> np.ndarray:
    print(f'点群取得中: {url}')
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    points = np.array(resp.json()['points'], dtype=np.float64)
    print(f'取得完了: {len(points)} 点')
    return points


# ── 検証 ─────────────────────────────────────────────────────────────────────

def main() -> None:
    sensor_url = f'{API_HOST}:{API_PORT}/api/v1/samples/{SAMPLE_TOKEN}/sensor-data'
    sensor_data = requests.get(sensor_url).json()['LIDAR_TOP']
    ego_pose_url = f'{API_HOST}:{API_PORT}/api/v1/sensor-data/{sensor_data["token"]}/ego-pose'
    ego_z = requests.get(ego_pose_url).json()['translation'][2]
    print(f'自車の地面からの高さ (ego_pose.translation[2]): {ego_z:.3f} m')
    calibrated_sensor_url = f'{API_HOST}:{API_PORT}/api/v1/calibrated-sensors/{sensor_data["calibrated_sensor_token"]}'
    lidar_height = requests.get(calibrated_sensor_url).json()['translation'][2]
    print(f'LIDAR_TOP の地面からの高さ (calibrated_sensor.translation[2]): {lidar_height:.3f} m')
    pointcloud_url = f'{API_HOST}:{API_PORT}/api/v1/sensor-data/{sensor_data["token"]}/pointcloud'
    points = fetch_pointcloud(pointcloud_url)

    # pointcloudのzの生値のヒストグラムをプロット
    plt.figure(figsize=(10, 5))
    plt.hist(points[:, 2], bins=100, color='blue', alpha=0.7)
    plt.title('Raw Point Cloud Z Histogram')
    plt.xlabel('Z (m)')
    plt.ylabel('Frequency')
    plt.xlim(-5, 5)  # zの範囲を制限して見やすくする
    plt.show()

    # ── 検証 1: 複数のターゲット位置で推定 ────────────────────────────────────
    # 自車近傍 / 前後左右 / 遠方 を含むグリッド
    targets = [
        (0.0,   0.0),    # 自車直下
        (5.0,   0.0),    # 前方 5m (センサー座標の +x が前とは限らないが目安)
        (-5.0,  0.0),
        (0.0,   5.0),
        (0.0,  -5.0),
        (10.0,  10.0),
        (-10.0, -10.0),
        (20.0,  0.0),    # 遠方
        (0.0,   30.0),   # さらに遠方 (点が疎な可能性)
        (40.0,  40.0),   # おそらく点が無い → prior フォールバック確認
    ]

    print('\n── 各ターゲット位置での推定結果 ──')
    print(f'{"target (x, y)":>18} | {"ground_z":>9} | method')
    print('-' * 55)
    results = []
    for txy in targets:
        gz, method = estimate_ground_z(points, txy, lidar_height, ego_z)
        results.append((txy, gz, method))
        print(f'({txy[0]:7.1f}, {txy[1]:7.1f}) | {gz:9.3f} | {method}')

    # ── 検証 2: 自車周辺の詳細ログ ─────────────────────────────────────────────
    print('\n── 自車直下 (0, 0) の詳細 ──')
    estimate_ground_z(points, (0.0, 0.0), lidar_height, ego_z, verbose=True)

    # ── 検証 3: 可視化 ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 3-1: BEV (上から見た点群) + ターゲット位置
    ax = axes[0]
    # 描画負荷軽減のため間引き
    step = max(1, len(points) // 30000)
    sub = points[::step]
    sc = ax.scatter(sub[:, 0], sub[:, 1], c=sub[:, 2], s=0.5,
                    cmap='viridis', vmin=-lidar_height - 0.5, vmax=-lidar_height + 2.5)
    for (txy, gz, method) in results:
        color = 'red' if method.startswith('estimated') else 'orange'
        ax.plot(txy[0], txy[1], 'x', color=color, markersize=10, markeredgewidth=2)
    ax.set_title('BEV (color = z) / x=estimated, orange=prior fallback')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, label='z (m)')

    # 3-2: 自車近傍 (R=3m) の z ヒストグラム
    ax = axes[1]
    z_min = -lidar_height - LOWER_MARGIN
    z_max = lidar_height + UPPER_MARGIN
    d2 = points[:, 0] ** 2 + points[:, 1] ** 2
    near = points[(d2 <= 9.0) & (points[:, 2] >= z_min) & (points[:, 2] <= z_max)]
    ax.hist(near[:, 2], bins=int((z_max - z_min) / BIN_SIZE), range=(z_min, z_max),
            edgecolor='black', alpha=0.7)
    gz0, _ = estimate_ground_z(points, (0.0, 0.0), lidar_height, ego_z)
    ax.axvline(gz0, color='red', linestyle='--', label=f'estimated = {gz0:.3f}')
    ax.axvline(-lidar_height, color='orange', linestyle=':', label=f'prior = {-lidar_height:.2f}')
    ax.set_title('z histogram near ego (R=3m)')
    ax.set_xlabel('z (m)'); ax.set_ylabel('count')
    ax.legend()

    # 3-3: 側面図 (x-z 断面, |y| < 2m の点) + 推定地面ライン
    ax = axes[2]
    slice_pts = points[np.abs(points[:, 1]) < 2.0]
    ax.scatter(slice_pts[:, 0], slice_pts[:, 2], s=0.5, alpha=0.5)
    # x 方向に沿って推定地面をプロット
    xs = np.arange(-30, 31, 2.0)
    gzs = [estimate_ground_z(points, (float(x), 0.0), lidar_height, ego_z)[0] for x in xs]
    ax.plot(xs, gzs, 'r.-', label='estimated ground')
    ax.axhline(-lidar_height, color='orange', linestyle=':', label='prior')
    ax.set_title('side view (|y| < 2m) + estimated ground line')
    ax.set_xlabel('x (m)'); ax.set_ylabel('z (m)')
    ax.set_ylim(-lidar_height - 1.5, -lidar_height + 4.0)
    ax.legend()

    plt.tight_layout()
    out_path = 'ground_estimation_check.png'
    plt.savefig(out_path, dpi=120)
    print(f'\n可視化を保存: {out_path}')
    plt.show()


if __name__ == '__main__':
    main()